# Search/validation scores and Order-FNR associations


Recalculates search and validation scores from stored predicted matrices using their stage-specific thresholds. It then reconstructs Order-FNR from stored priors and tests its descriptive association with held-out performance.


In [1]:
from pathlib import Path
import sys, os, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "experiments/run_model.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "experiments")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import final_visualization as fv
import result_tables as th
fv.apply_style()
from metrics import compute_metrics
from scipy.stats import spearmanr


## Scores and Order-FNR associations


In [2]:
fits = []
recorded = th.load_results("model_selection")
for stage, name, threshold in [
    ("search", "search_trial_0068.npz", 0.25),
    ("validation", "validation_candidate_0068.npz", 0.1),
]:
    with np.load(ROOT / "results/prism/tuning/predictions" / name, allow_pickle=False) as matrices:
        for key in matrices.files:
            family, identity = key.split("__ds")
            ds, ms = map(int, identity.split("__ms"))
            with th.load_sample("model_selection", family, ds) as data:
                truth = data["true_adj"]
                labels = data["col_names"].astype(str).tolist()
            graph = (np.abs(matrices[key]) > threshold).astype(int)
            np.fill_diagonal(graph, 0)
            row = compute_metrics(truth, graph, labels)
            p, r = row["precision"], row["recall"]
            row["fbeta"] = 1.25 * p * r / (0.25 * p + r) if 0.25 * p + r else 0.0
            identity_row = recorded.loc[
                recorded.dataset.eq(family) & recorded.condition_id.eq(stage)
                & recorded.dataset_seed.eq(ds) & recorded.model_seed.eq(ms)
            ].iloc[0].to_dict()
            fits.append(
                dict(
                    **{key: value for key, value in identity_row.items() if key not in row and key not in ("dataset", "dataset_seed", "model_seed")},
                    stage=stage,
                    dataset=family,
                    dataset_seed=ds,
                    model_seed=ms,
                    threshold=threshold,
                    **row,
                )
            )
sources = th.load_aligned_sources()
complete = sources["complete"]
final = complete.loc[
    complete.model.eq("PRISM") & complete.dataset.isin(th.TEST_FAMILIES)
].copy()
final["stage"] = "test"
final["threshold"] = 0.1
stages = pd.concat([pd.DataFrame(fits), final], ignore_index=True)
blocks = stages.groupby(["stage", "dataset", "dataset_seed"]).fbeta.mean().reset_index()
macro = blocks.groupby(["stage", "dataset_seed"]).fbeta.mean().reset_index()
stage_summary = macro.groupby("stage").fbeta.agg(["mean", "std", "size"]).reset_index()


In [3]:
scores = th.seed_means(complete.loc[complete.model.eq("PRISM")], "fbeta", arms=("tuned",))
orders = []
for row in scores.itertuples(index=False):
    family, seed = row.dataset, int(row.dataset_seed)
    with th.load_sample("evaluation", family, seed) as d:
        truth = d["true_adj"]
    with np.load(th.ALIGNED_PRIORS / f"{family}__seed_{seed}.npz", allow_pickle=False) as d:
        order = d["order"]
    fnr = th.order_fnr(truth, order)
    orders.append(dict(dataset=family, dataset_seed=seed, fbeta=row.fbeta, order_fnr=fnr))
orders = pd.DataFrame(orders)
associations = []
for family, group in orders.groupby("dataset"):
    result = spearmanr(group.order_fnr, group.fbeta)
    associations.append(
        dict(
            dataset=family,
            n_pairs=len(group),
            spearman_rho=float(result.statistic),
            pvalue=float(result.pvalue),
        )
    )
correlations = pd.DataFrame(associations)


In [4]:
display(stage_summary)
display(correlations)


,stage,mean,std,size
0,search,0.587956,0.050211,5
1,test,0.614231,0.058831,15
2,validation,0.632772,0.061484,5


,dataset,n_pairs,spearman_rho,pvalue
0,20nodes_EB_obsconf0_latconf0_Gaussian_Linear,15,-0.580193,0.023364
1,20nodes_EB_obsconf0_latconf0_Gaussian_Mixed,15,-0.725000,0.002227
2,20nodes_EB_obsconf0_latconf0_NonGaussian_Linear,15,-0.738159,0.001677
3,20nodes_EB_obsconf0_latconf0_Non_Gaussian_Mixed,15,-0.375000,0.168433
4,20nodes_ModMultiOmics_obsconf0_latconf0_Laplac...,15,-0.889286,0.000009
5,20nodes_ModOncogene_obsconf0_latconf0_Gaussian...,15,-0.765654,0.000877


## Held-out performance and estimated causal order


In [5]:
prism_table = th.prism_performance(complete)
display(prism_table[["Split", "Dataset family", "F0.5", "Order FNR"]])


,Split,Dataset family,F0.5,Order FNR
0,Test,"ER, non Gaussian mixed",0.645 ± 0.087,0.185 ± 0.091
1,Test,"MultiOmics, Laplace mixed",0.584 ± 0.138,0.288 ± 0.153
2,Test,"Oncogene, Laplace mixed",0.613 ± 0.092,0.256 ± 0.095
3,OOD,"ER, Gaussian linear",0.475 ± 0.162,0.062 ± 0.133
4,OOD,"ER, Gaussian mixed",0.771 ± 0.082,0.093 ± 0.066
5,OOD,"ER, non Gaussian linear",0.438 ± 0.155,0.109 ± 0.156


## PRISM search and training parameters

The search uses one 400-trial budget. The package calls the constant gate `none` and the subsampling direction formulation `bootstrap`.


In [6]:
config = json.loads((ROOT / "configs/prism_study.json").read_text())
winner = json.loads((ROOT / "configs/prism_winner.json").read_text())
parameters = pd.DataFrame([dict(parameter=name, selected=winner["params"][name],
                                search=json.dumps(definition)) for name, definition in config["hpo"]["space"].items()])
display(parameters)
training = pd.DataFrame([dict(parameter=name, value=config["training"][name]) for name in
                         ("hidden_dim", "T", "lr", "lambda2", "warm_iter", "max_iter")])
display(training)


,parameter,selected,search
0,alpha,0.501815,"{""high"": 1.0, ""low"": 0.0, ""type"": ""float""}"
1,direction,binary,"{""choices"": [""binary"", ""bootstrap""], ""type"": ""..."
2,gate,none,"{""choices"": [""none"", ""linear"", ""floor20percent..."
3,lambda_d,2.317528,"{""high"": 3.0, ""low"": 0.0001, ""type"": ""float_log""}"
4,lambda_e,0.253194,"{""high"": 0.3, ""low"": 0.0001, ""type"": ""float_log""}"
5,lambda_s,0.000276,"{""high"": 0.01, ""low"": 1e-06, ""type"": ""float_log""}"
6,q_G,0.71385,"{""high"": 10.0, ""low"": 0.1, ""type"": ""float_log""}"
7,q_H,0.420773,"{""high"": 10.0, ""low"": 0.1, ""type"": ""float_log""}"


,parameter,value
0,hidden_dim,8.000000
1,T,6.000000
2,lr,0.001043
3,lambda2,0.000544
4,warm_iter,3125.000000
5,max_iter,5000.000000
